# Chapter 7 — Character-Level Tokenization

Chapter 6 selected a lightly prepared text that preserves capitalization, punctuation, and paragraph structure.
This chapter builds a reusable tokenizer that converts that text into integer IDs and back again.

By the end of this chapter, you will be able to:

- explain character-level tokenization;
- build a deterministic character vocabulary and inverse mappings;
- encode compatible text as character IDs;
- decode character IDs back into exact text;
- verify complete round trips;
- explain why IDs are labels and vocabulary order is part of the tokenizer;
- fail clearly on unknown characters and IDs; and
- describe the main tradeoffs of character-level tokenization.

## The Tokenizer Contract

A **tokenizer** defines the units used to represent text and the rules for converting between text and IDs.
A **character token** is one Python string element treated as a token.
This course continues Chapter 4's simplified use of Python characters rather than addressing every Unicode grapheme complication.

A **vocabulary** is the ordered collection of tokens known to the tokenizer.
A **token ID** is the integer label assigned to one vocabulary entry.
**Encoding** converts text into IDs, while **decoding** converts IDs into text.
A **round trip** succeeds when decoding encoded text reproduces the original text exactly.

The vocabulary and both conversions form one contract.
Changing the vocabulary order changes that contract even when the vocabulary contains the same characters.

## Use the Prepared Text from Chapter 6

The fixture has metadata removed and accidental horizontal whitespace normalized.
Its capitalization, punctuation, line breaks, and paragraph break remain.
The `repr` view confirms that the blank line is represented by two consecutive newline characters.

In [1]:
prepared_text = (
    "The dog ran across the yard.\n"
    "The cat sat near the window.\n"
    "\n"
    "The dog looked at the cat!\n"
    "The cat looked back."
)

print("Prepared text:")
print(prepared_text)
print()
print("Technical representation:")
print(repr(prepared_text))

Prepared text:
The dog ran across the yard.
The cat sat near the window.

The dog looked at the cat!
The cat looked back.

Technical representation:
'The dog ran across the yard.\nThe cat sat near the window.\n\nThe dog looked at the cat!\nThe cat looked back.'


## Turn the String into Character Tokens

Iterating over a Python string yields its characters in order.
Calling `list` makes those character tokens explicit.
The token count therefore equals the string length for this tokenizer.

The position table uses `repr` so spaces and newlines remain visible.

In [2]:
character_tokens = list(prepared_text)

print("Token count:", len(character_tokens))
print("Text length:", len(prepared_text))
print("Counts match:", len(character_tokens) == len(prepared_text))
print()
print("Position | Character")
print("-" * 24)
for position, character in enumerate(character_tokens[:40]):
    print(f"{position:>8} | {character!r}")

Token count: 106
Text length: 106
Counts match: True

Position | Character
------------------------
       0 | 'T'
       1 | 'h'
       2 | 'e'
       3 | ' '
       4 | 'd'
       5 | 'o'
       6 | 'g'
       7 | ' '
       8 | 'r'
       9 | 'a'
      10 | 'n'
      11 | ' '
      12 | 'a'
      13 | 'c'
      14 | 'r'
      15 | 'o'
      16 | 's'
      17 | 's'
      18 | ' '
      19 | 't'
      20 | 'h'
      21 | 'e'
      22 | ' '
      23 | 'y'
      24 | 'a'
      25 | 'r'
      26 | 'd'
      27 | '.'
      28 | '\n'
      29 | 'T'
      30 | 'h'
      31 | 'e'
      32 | ' '
      33 | 'c'
      34 | 'a'
      35 | 't'
      36 | ' '
      37 | 's'
      38 | 'a'
      39 | 't'


## Build a Deterministic Vocabulary

The character vocabulary contains each unique character in the prepared text exactly once.
Sorting gives the same order whenever the same set of characters is used.
The sorted order is deterministic rather than linguistic or semantic.

The vocabulary includes invisible characters because they are part of the prepared text.

In [3]:
def build_character_vocabulary(text: str) -> list[str]:
    return sorted(set(text))


character_vocabulary = build_character_vocabulary(prepared_text)

print("ID | Character")
print("-" * 18)
for character_id, character in enumerate(character_vocabulary):
    print(f"{character_id:>2} | {character!r}")
print("Vocabulary size:", len(character_vocabulary))

ID | Character
------------------
 0 | '\n'
 1 | ' '
 2 | '!'
 3 | '.'
 4 | 'T'
 5 | 'a'
 6 | 'b'
 7 | 'c'
 8 | 'd'
 9 | 'e'
10 | 'g'
11 | 'h'
12 | 'i'
13 | 'k'
14 | 'l'
15 | 'n'
16 | 'o'
17 | 'r'
18 | 's'
19 | 't'
20 | 'w'
21 | 'y'
Vocabulary size: 22


## Build Inverse Mappings

The encoder needs a character-to-ID mapping.
The decoder needs the inverse ID-to-character mapping.

The function rejects empty vocabularies and duplicate entries because neither defines a useful one-to-one tokenizer mapping.
The checks afterward verify that every character maps to an ID and back to itself.

In [4]:
def build_character_mappings(
    vocabulary: list[str],
) -> tuple[dict[str, int], dict[int, str]]:
    if not vocabulary:
        raise ValueError("The vocabulary must not be empty.")
    if len(vocabulary) != len(set(vocabulary)):
        raise ValueError("The vocabulary must not contain duplicates.")

    character_to_id = {
        character: character_id for character_id, character in enumerate(vocabulary)
    }
    id_to_character = {
        character_id: character for character, character_id in character_to_id.items()
    }
    return character_to_id, id_to_character


character_to_id, id_to_character = build_character_mappings(character_vocabulary)

for character in character_vocabulary:
    assert id_to_character[character_to_id[character]] == character

print("All vocabulary entries passed the mapping round trip.")
print("Character to ID:")
print(character_to_id)

All vocabulary entries passed the mapping round trip.
Character to ID:
{'\n': 0, ' ': 1, '!': 2, '.': 3, 'T': 4, 'a': 5, 'b': 6, 'c': 7, 'd': 8, 'e': 9, 'g': 10, 'h': 11, 'i': 12, 'k': 13, 'l': 14, 'n': 15, 'o': 16, 'r': 17, 's': 18, 't': 19, 'w': 20, 'y': 21}


## Token IDs Are Labels

A character ID identifies one vocabulary entry.
Its numerical size does not express importance, similarity, frequency, or meaning.

The table below connects text positions with characters and their arbitrary integer labels.
Later embeddings will associate IDs with learned vectors, but the raw IDs themselves remain labels.

In [5]:
print("Position | Character | Character ID")
print("-" * 38)
for position, character in enumerate(prepared_text[:30]):
    character_id = character_to_id[character]
    print(f"{position:>8} | {character!r:>9} | {character_id:>12}")

Position | Character | Character ID
--------------------------------------
       0 |       'T' |            4
       1 |       'h' |           11
       2 |       'e' |            9
       3 |       ' ' |            1
       4 |       'd' |            8
       5 |       'o' |           16
       6 |       'g' |           10
       7 |       ' ' |            1
       8 |       'r' |           17
       9 |       'a' |            5
      10 |       'n' |           15
      11 |       ' ' |            1
      12 |       'a' |            5
      13 |       'c' |            7
      14 |       'r' |           17
      15 |       'o' |           16
      16 |       's' |           18
      17 |       's' |           18
      18 |       ' ' |            1
      19 |       't' |           19
      20 |       'h' |           11
      21 |       'e' |            9
      22 |       ' ' |            1
      23 |       'y' |           21
      24 |       'a' |            5
      25 |       'r' |   

## Encode Text as IDs

Encoding replaces each character with its assigned ID while preserving sequence order.
The function checks membership before lookup so an unknown character produces a precise error with its position.
Silent skipping would corrupt the text and change every later position.

In [6]:
def encode_text(
    text: str,
    character_to_id: dict[str, int],
) -> list[int]:
    character_ids = []

    for position, character in enumerate(text):
        if character not in character_to_id:
            raise ValueError(
                f"Unknown character {character!r} at text position {position}."
            )
        character_ids.append(character_to_id[character])

    return character_ids


character_ids = encode_text(prepared_text, character_to_id)

print("First 60 character IDs:")
print(character_ids[:60])
print("ID count matches text length:", len(character_ids) == len(prepared_text))

First 60 character IDs:
[4, 11, 9, 1, 8, 16, 10, 1, 17, 5, 15, 1, 5, 7, 17, 16, 18, 18, 1, 19, 11, 9, 1, 21, 5, 17, 8, 3, 0, 4, 11, 9, 1, 7, 5, 19, 1, 18, 5, 19, 1, 15, 9, 5, 17, 1, 19, 11, 9, 1, 20, 12, 15, 8, 16, 20, 3, 0, 0, 4]
ID count matches text length: True


## Decode IDs as Text

Decoding performs the inverse replacement and joins the recovered characters.
The decoder also reports the sequence position of an unknown ID.

Exact equality verifies that spaces, punctuation, capitalization, and newlines all survived the complete round trip.

In [7]:
def decode_ids(
    character_ids: list[int],
    id_to_character: dict[int, str],
) -> str:
    characters = []

    for position, character_id in enumerate(character_ids):
        if character_id not in id_to_character:
            raise ValueError(
                f"Unknown character ID {character_id} at sequence position {position}."
            )
        characters.append(id_to_character[character_id])

    return "".join(characters)


decoded_text = decode_ids(character_ids, id_to_character)

print("Decoded text matches prepared text:", decoded_text == prepared_text)
print("Decoded technical representation:")
print(repr(decoded_text))

assert decoded_text == prepared_text

Decoded text matches prepared text: True
Decoded technical representation:
'The dog ran across the yard.\nThe cat sat near the window.\n\nThe dog looked at the cat!\nThe cat looked back.'


## Package the Contract in a Small Class

The standalone functions exposed every part of the tokenizer before abstraction.
A small class now keeps one vocabulary and its mappings together so later notebooks cannot accidentally mix them.

The constructor accepts an ordered vocabulary rather than rebuilding one silently.
That design makes the vocabulary an explicit artifact that can be preserved with a future model.

In [8]:
class CharacterTokenizer:
    vocabulary: list[str]
    character_to_id: dict[str, int]
    id_to_character: dict[int, str]

    def __init__(self, vocabulary: list[str]) -> None:
        self.vocabulary = vocabulary.copy()
        self.character_to_id, self.id_to_character = build_character_mappings(
            self.vocabulary
        )

    def encode(self, text: str) -> list[int]:
        return encode_text(text, self.character_to_id)

    def decode(self, character_ids: list[int]) -> str:
        return decode_ids(character_ids, self.id_to_character)


character_tokenizer = CharacterTokenizer(character_vocabulary)
tokenizer_round_trip = character_tokenizer.decode(
    character_tokenizer.encode(prepared_text)
)

print("Vocabulary size:", len(character_tokenizer.vocabulary))
print("Class round trip succeeded:", tokenizer_round_trip == prepared_text)
assert tokenizer_round_trip == prepared_text

Vocabulary size: 22
Class round trip succeeded: True


## Encode Compatible New Text

The tokenizer is not limited to encoding its complete training fixture.
It can encode any new string whose characters all belong to its vocabulary.
The short example below verifies that property with another exact round trip.

In [9]:
compatible_text = "The cat sat."
compatible_ids = character_tokenizer.encode(compatible_text)
recovered_compatible_text = character_tokenizer.decode(compatible_ids)

print("Text:", repr(compatible_text))
print("IDs:", compatible_ids)
print("Recovered text:", repr(recovered_compatible_text))
assert recovered_compatible_text == compatible_text

Text: 'The cat sat.'
IDs: [4, 11, 9, 1, 7, 5, 19, 1, 18, 5, 19, 3]
Recovered text: 'The cat sat.'


## Fail Clearly on Unknown Values

An **unknown character** is absent from the fixed vocabulary.
An **unknown ID** is absent from the inverse mapping.

The prepared text contains neither lowercase `z` nor an ID equal to the vocabulary size.
The next cell catches both intentional errors so their messages remain visible without stopping the notebook.
Later tokenizers may use unknown tokens, bytes, or subwords instead of failing.

In [10]:
try:
    character_tokenizer.encode("The zebra ran.")
except ValueError as error:
    print("Expected encoding error:")
    print(error)

unknown_character_id = len(character_tokenizer.vocabulary)

try:
    character_tokenizer.decode([unknown_character_id])
except ValueError as error:
    print()
    print("Expected decoding error:")
    print(error)

Expected encoding error:
Unknown character 'z' at text position 4.

Expected decoding error:
Unknown character ID 22 at sequence position 0.


## Preserve Vocabulary Order

A trained model associates each output position with one token ID.
Reordering the same characters therefore changes what those positions mean.

The example builds a second tokenizer from the reversed vocabulary.
Both tokenizers know the same characters, but they encode the same text differently.
A future model and tokenizer must always use the same preserved vocabulary order.

In [11]:
reversed_vocabulary = list(reversed(character_vocabulary))
reordered_tokenizer = CharacterTokenizer(reversed_vocabulary)
comparison_text = "The"

original_order_ids = character_tokenizer.encode(comparison_text)
reversed_order_ids = reordered_tokenizer.encode(comparison_text)

print("Comparison text:", repr(comparison_text))
print("IDs with original order:", original_order_ids)
print("IDs with reversed order:", reversed_order_ids)
print("The ID sequences differ:", original_order_ids != reversed_order_ids)

assert original_order_ids != reversed_order_ids

Comparison text: 'The'
IDs with original order: [4, 11, 9]
IDs with reversed order: [17, 10, 12]
The ID sequences differ: True


## Understand the Tradeoffs

Character-level tokenization is simple, exact for known characters, and able to represent unseen words composed of known characters.
Its small vocabulary also makes each model output easier to inspect.

The main cost is sequence length because every character becomes a separate token.
A simple whitespace split produces fewer tokens but loses exact spacing and does not solve punctuation or unknown-word problems.
The count comparison illustrates sequence length without claiming that the word split is a better tokenizer.

In [12]:
simple_word_tokens = prepared_text.split()

print("Character token count:", len(character_tokens))
print("Simple word token count:", len(simple_word_tokens))
print("First 20 character tokens:", character_tokens[:20])
print("First 20 simple word tokens:", simple_word_tokens[:20])

Character token count: 106
Simple word token count: 22
First 20 character tokens: ['T', 'h', 'e', ' ', 'd', 'o', 'g', ' ', 'r', 'a', 'n', ' ', 'a', 'c', 'r', 'o', 's', 's', ' ', 't']
First 20 simple word tokens: ['The', 'dog', 'ran', 'across', 'the', 'yard.', 'The', 'cat', 'sat', 'near', 'the', 'window.', 'The', 'dog', 'looked', 'at', 'the', 'cat!', 'The', 'cat']


## Final Checks

The final cell gathers the tokenizer's essential invariants.
These checks protect later notebooks from a mismatched vocabulary, corrupted mapping, or lossy conversion.

In [13]:
assert len(character_tokens) == len(prepared_text)
assert len(character_vocabulary) == len(set(character_vocabulary))
assert set(character_vocabulary) == set(prepared_text)
assert set(character_to_id.values()) == set(id_to_character)
assert (
    character_tokenizer.decode(character_tokenizer.encode(prepared_text))
    == prepared_text
)
assert (
    character_tokenizer.decode(character_tokenizer.encode("The cat sat."))
    == "The cat sat."
)

print("All character-tokenizer checks passed.")
print("Vocabulary size:", len(character_vocabulary))
print("Prepared sequence length:", len(character_ids))

All character-tokenizer checks passed.
Vocabulary size: 22
Prepared sequence length: 106


## Key Takeaways

- Character-level tokenization assigns one token to each Python character in order.
- A deterministic ordered vocabulary defines the meaning of every token ID.
- Character-to-ID and ID-to-character mappings must be exact inverses.
- Encoding and decoding should reproduce compatible text without any loss.
- Unknown characters and IDs should fail explicitly rather than alter data silently.
- Token IDs are labels without numerical language meaning.
- Character tokenizers are easy to inspect but create longer sequences than coarser tokenizers.
- A future model must use the same preserved vocabulary order used during training.

## Next Chapter

Chapter 8 introduces word-level tokenization to contrast shorter sequences with a much larger and less flexible vocabulary.
That comparison will make punctuation handling, exact spacing, and unknown-word behavior concrete before the course moves toward subword tokenization.